In [2]:
# Import packages

%matplotlib qt
from matplotlib.pyplot import plot, show, title

import scipy
import scipy.signal

import numpy
from numpy import loadtxt
from numpy import savetxt

import biosppy
from biosppy.signals import ecg

from opensignalsreader import OpenSignalsReader

import pyhrv.tools as tools
from pyhrv.hrv import hrv

# ECG file upload

fs - Recording ECG frequency. Swell - 2048 Hz; Wesad - 700 Hz

In [3]:
#/media/eduarda/Seagate/RawFilteredSwellData/S1_stress_I.csv
#/media/eduarda/Seagate/RawWesad/S2_baseline_ECG_values.csv
ecg = loadtxt("/media/eduarda/Seagate/RawFilteredSwellData/S1_neutral.csv", delimiter=',')

#Swell - 2048Hz; Wesad - 700Hz
fs = 2048

# Filtering ButterWorth Filter (Wesad: 0.5-40Hz)

In [4]:
def bandpassfilter(signal, fs):
    lowcut=0.5 #8
    highcut=40 #55
    
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    
    order = 2
    
    b, a = scipy.signal.butter(order, [low,high], 'bandpass', analog=False)
    y = scipy.signal.filtfilt(b,a, signal, axis=0)
    
    return(y)

In [5]:
#--------------WESAD--------------------

#filtered_ecg = bandpassfilter(ecg, fs)

#--------------SWELL--------------------

filtered_ecg=ecg

In [7]:
#number of samples in the .csv file, i.e. number of data points
print(filtered_ecg.size)

7219696


In [8]:
#S1_stress_I (c3): duration 3367000 (ms)
#S2_baseline: duration 1518312 (ms)

ecg_duration = 3525 #s
samples = 7219696 # = filtered_ecg.size
time = ecg_duration / samples # time between consecutive data points in the .csv file

## Plotting ECG: Original and Filtered ; Saving filtered file optional

In [9]:
#title("Original signal")
#plot(ecg)
#show()

#title("Filtered signal")
#plot(data)
#show()

#from numpy import savetxt
#savetxt( "S2_stress_ECG_values_FILTERED.csv", data, delimiter=',')

# Detecting R Peaks with Biospy (Hamilton)

https://biosppy.readthedocs.io/en/stable/biosppy.signals.html#biosppy-signals-ecg

In [10]:
def rPeakHamiltonTempo(filtered_ecg, fs, time):
    #Returns R-peak location indices (sample number, IT IS NOT TIME)
    rPeakLocation = biosppy.signals.ecg.hamilton_segmenter(signal=filtered_ecg, sampling_rate=fs)
    
    #Obtaining R-peak values in time (ms): R-peak sample number -> R-peak time value
    tt = []
    
    for i in rPeakLocation:
        xx = i * time
        tt.append(xx)
        i = i + 1
    
    print (tt)
    
    return tt

# Detecting R Peaks with default parameters

https://biosppy.readthedocs.io/en/stable/biosppy.signals.html#biosppy-signals-ecg

In [11]:
def rPeak(filtered_ecg, fs, time):
    #Returns R-peak location indices (sample number, IT IS NOT TIME)
    rPeakLocation = biosppy.signals.ecg.ecg(filtered_ecg, sampling_rate=fs)[2]
    
    #Obtaining R-peak values in time (ms): R-peak sample number -> R-peak time value
    tt = []
    
    for i in rPeakLocation:
        xx = i * time
        tt.append(xx)
        i = i + 1
    
    print (tt)
    
    return tt

## The variable `rPeak` returns the time in which the R-peaks occured

In [12]:
#R-peak Hamilton
#rPeaks = rPeakHamiltonTempo(filtered_ecg, fs, time)

#R-peaks default parameters
rPeaks = rPeak(filtered_ecg, fs, time)

/home/eduarda/anaconda3/lib/python3.7/site-packages/biosppy/plotting.py:1145: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  gs.tight_layout(fig)


[400.85136548685705, 1269.9322796970953, 1884.147892099612, 2734.6753935345755, 3544.1900877820894, 4346.869314165028, 5145.1543112064555, 5943.927555952495, 6763.207204292258, 8732.798444699058, 10123.81615514005, 10693.60122642283, 11230.673701496573, 11785.323093936366, 12387.820761428182, 13194.894217152634, 13985.36725092026, 14749.474908638813, 16082.39114223092, 16917.294717118282, 17742.92158561801, 18554.87751838859, 19338.515084291637, 20113.852439216276, 20858.918436454944, 21582.013286986046, 22289.97245867416, 22974.007492836263, 23664.38974715833, 24325.477139203645, 25385.95115362198, 25763.36662928744, 26382.952966440695, 27004.98054211701, 27622.125640747203, 28229.994032989754, 28818.33251704781, 29402.27677176435, 29985.73277877628, 30562.84156562825, 31139.462104775605, 31711.20016687683, 32284.891219796515, 32858.582272716194, 33427.39084858975, 34275.47711150165, 34828.173513122994, 35377.94042851666, 35935.031059479516, 36494.07468126082, 37002.34054176242, 37613

/home/eduarda/anaconda3/lib/python3.7/site-packages/ipykernel/eventloops.py:106: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  app.exec_()


# Calculating RR intervals (RR)

In [16]:
def detectrr(rPeaks):
    
    #Obtaining RR intervals (difference in time between consecutive R-Peaks)
    rr = []
    for i in range(len(rPeaks) - 1):
        xx = rPeaks[i + 1] - rPeaks[i]
        rr.append(xx)
        i = i + 1
    return rr

In [17]:
rr_interval = detectrr(rPeaks)
print (rr_interval)

[869.0809142102382, 614.2156124025166, 850.5275014349636, 809.514694247514, 802.6792263829389, 798.2849970414272, 798.7732447460394, 819.2796483397633, 1969.5912404067994, 1391.0177104409922, 569.7850712827803, 537.0724750737427, 554.6493924397928, 602.4976674918162, 807.0734557244523, 790.4730337676265, 764.1076577185522, 1332.9162335921064, 834.903574887363, 825.6268684997267, 811.9559327705792, 783.6375659030491, 775.3373549246389, 745.0659972386675, 723.0948505311026, 707.959171688115, 684.0350341621015, 690.3822543220667, 661.0873920453159, 1060.4740144183343, 377.4154756654607, 619.5863371532541, 622.0275756763149, 617.1450986301934, 607.8683922425516, 588.3384840580547, 583.9442547165418, 583.4560070119296, 577.108786851968, 576.6205391473559, 571.7380621012271, 573.691052919683, 573.6910529196794, 568.8085758735542, 848.0862629119001, 552.696401621346, 549.7669153936658, 557.0906309628554, 559.043621781304, 508.26586050160404, 610.7978784702282, 577.5970345565802, 583.456007011

# HRV Calculation

https://pyhrv.readthedocs.io/en/latest/_pages/api/hrv.html

In [18]:
#Calculating HRV based on rPeaks calculated before
hrv = hrv(rpeaks=rPeaks, sampling_rate=fs)

#Calculating HRV with filtered ecg, default parameters
#hrv = hrv(signal=filtered_ecg, sampling_rate=fs)

/home/eduarda/anaconda3/lib/python3.7/site-packages/pyhrv/time_domain.py:576: UserWarning: CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a malfunction of the function. This function will be reviewed over the next updates to solve this issue
  warnings.warn('CAUTION: The TINN computation is currently providing incorrect results in the most cases due to a '


In [19]:
# the variable `hrv` is classbiosppy.utils.ReturnTuple(values, names=None), must be converted to ordered dictionary
hrv_dic = hrv.as_dict()

In [20]:
mean_rr = hrv_dic["hr_mean"]
print("Mean Heart Rate: " , mean_rr)

sdnn = hrv_dic["sdnn"]
print("SDNN: " , sdnn)

rmssd = hrv_dic["rmssd"]
print("RMSSD: " , rmssd)

sdann = hrv_dic["sdann"]
print("SDANN: " , sdann)

Mean Heart Rate:  68.2253974404475
SDNN:  100.9520385964312
RMSSD:  81.6052919098206
SDANN:  17.108399636016706
